# K-Nearest Neighbors Music Recommendation System

### Overview
This system implements a K-Nearest Neighbors (KNN) approach with:
- **Interactive Menu System** for dynamic queue management
- **Artist & Genre Similarity** using one-hot encoding
- **Cosine Similarity** for recommendations
- **Real-time Updates** as you modify your queue

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
from scipy.spatial.distance import cosine
import warnings
warnings.filterwarnings('ignore')

print(" Libraries imported successfully!")

## Step 1: Load Dataset

In [ ]:
# Load dataset
df = pd.read_csv('Data.csv')

print("DATASET LOADED SUCCESSFULLY!")
print(f" Total songs in dataset: {len(df)}")
print(f" Total features: {len(df.columns) - 1}")
print(f" Dataset shape: {df.shape}")

## Step 2: Data Preprocessing

In [ ]:
# Extract song names (first column)
song_names = df.iloc[:, 0].values

# Extract feature matrix (all columns except first)
feature_matrix = df.iloc[:, 1:].values

# Get feature column names dynamically
feature_columns = df.columns[1:].tolist()

print(f" Song names array shape: {song_names.shape}")
print(f" Feature matrix shape: {feature_matrix.shape}")
print(f"\n Feature matrix: {feature_matrix.shape[0]} songs × {feature_matrix.shape[1]} features")

## Step 3: Core Mathematical Functions

In [ ]:
def get_song_vector(song_name, song_names, feature_matrix):
    try:
        idx = np.where(song_names == song_name)[0][0]
        return feature_matrix[idx]
    except IndexError:
        return None


def compute_centroid(queue, song_names, feature_matrix):
    """
    Compute the centroid (mean vector) of songs in queue.
    
    Mathematical Formula:
    Centroid = (v₁ + v₂ + ... + vₙ) / n
    
    This represents the "average" characteristics of the queue.
    """
    vectors = []
    
    for song in queue:
        vec = get_song_vector(song, song_names, feature_matrix)
        if vec is not None:
            vectors.append(vec)
    
    if len(vectors) == 0:
        return None
    
    vectors = np.array(vectors)
    centroid = np.mean(vectors, axis=0)
    
    return centroid


def cosine_similarity(vec1, vec2):
    """
    Calculate cosine similarity between two vectors.
    
    Formula: sim(A,B) = (A·B) / (||A|| × ||B||)
    
    Returns value between 0 and 1:
    - 1.0 = Identical direction (very similar)
    - 0.0 = Orthogonal (no similarity)
    """
    return 1 - cosine(vec1, vec2)


def recommend_songs(queue, song_names, feature_matrix, k=5):
    """
    K-Nearest Neighbors recommendation algorithm.
    
    Steps:
    1. Compute centroid of current queue
    2. Calculate cosine similarity to all songs
    3. Sort by similarity (descending)
    4. Return top K songs not in queue
    
    Returns:
    - List of tuples: [(song_name, similarity_score), ...]
    """
    centroid = compute_centroid(queue, song_names, feature_matrix)
    
    if centroid is None:
        return []
    
    similarities = []
    
    for i, song in enumerate(song_names):
        if song in queue:
            continue
        
        song_vec = feature_matrix[i]
        similarity = cosine_similarity(centroid, song_vec)
        similarities.append((song, similarity))
    
    similarities.sort(key=lambda x: x[1], reverse=True)
    
    return similarities[:k]

print(" Core mathematical functions defined!")

## Step 4: Interactive Music Queue Class

In [ ]:
class InteractiveMusicQueue:
    
    def __init__(self, song_names, feature_matrix):
        self.song_names = song_names
        self.feature_matrix = feature_matrix
        self.queue = []
        self.all_songs_list = list(song_names)
    
    def display_menu(self):
        print("\n")
        print(" MUSIC RECOMMENDATION SYSTEM - MAIN MENU")
        print("1.  Add song to queue")
        print("2.  Remove song from queue")
        print("3.  View queue and recommendations")
        print("4.  Clear entire queue")
        print("5.  View all available songs")
        print("6.  Search for a song")
        print("7.  View detailed analysis")
        print("8.  Create random queue")
        print("9.  Load preset queue")
        print("0.  Exit")
    
    def add_song_interactive(self):
        print("\n")
        print(" ADD SONG TO QUEUE")
        
        if len(self.queue) >= len(self.song_names):
            print("  Queue is full! All songs are already added.")
            return
        
        print(f"Current queue size: {len(self.queue)}")
        print("\nOptions:")
        print("  1. Enter song name manually")
        print("  2. Browse and select from available songs")
        print("  3. Add from current recommendations")
        print("  0. Go back")
        
        choice = input("\nYour choice: ").strip()
        
        if choice == '1':
            song_name = input("\nEnter exact song name (e.g., 'Song Name - Artist'): ").strip()
            self.add_song(song_name)
        
        elif choice == '2':
            self.browse_and_add()
        
        elif choice == '3':
            self.add_from_recommendations()
        
        elif choice == '0':
            return
        
        else:
            print(" Invalid choice!")
    
    def browse_and_add(self):
        
        available_songs = [s for s in self.all_songs_list if s not in self.queue]
        
        if not available_songs:
            print("\n  No songs available to add!")
            return
        
        print("\n")
        print(f"AVAILABLE SONGS (Showing first 20)")
        
        display_count = min(20, len(available_songs))
        for i in range(display_count):
            print(f"{i+1:2d}. {available_songs[i]}")
        
        if len(available_songs) > 20:
            print(f"\n... and {len(available_songs) - 20} more songs")
        
        try:
            choice = input("\nEnter song number (or 0 to cancel): ").strip()
            if choice == '0':
                return
            
            idx = int(choice) - 1
            if 0 <= idx < display_count:
                self.add_song(available_songs[idx])
            else:
                print(" Invalid song number!")
        except ValueError:
            print(" Please enter a valid number!")
    
    def add_from_recommendations(self):
        
        if len(self.queue) == 0:
            print("\n  Queue is empty! Add some songs first to get recommendations.")
            return
        
        recommendations = recommend_songs(self.queue, self.song_names, self.feature_matrix, k=10)
        
        if not recommendations:
            print("\n  No recommendations available!")
            return
        
        print("\n")
        print("CURRENT RECOMMENDATIONS")
        
        for i, (song, score) in enumerate(recommendations, 1):
            print(f"{i:2d}. {song} (Score: {score:.4f})")
        
        try:
            choice = input("\nEnter recommendation number (or 0 to cancel): ").strip()
            if choice == '0':
                return
            
            idx = int(choice) - 1
            if 0 <= idx < len(recommendations):
                song_to_add = recommendations[idx][0]
                self.add_song(song_to_add)
            else:
                print(" Invalid recommendation number!")
        except ValueError:
            print(" Please enter a valid number!")
    
    def add_song(self, song_name):
    
        if song_name not in self.song_names:
            print(f"\n Error: '{song_name}' not found in dataset!")
            return False
        
        if song_name in self.queue:
            print(f"\n  '{song_name}' is already in the queue!")
            return False
        
        self.queue.append(song_name)
        print(f"\n Added: {song_name}")
        print(f" Queue size: {len(self.queue)}")
        return True
    
    def remove_song_interactive(self):
      
        if len(self.queue) == 0:
            print("\n  Queue is empty! Nothing to remove.")
            return
        
        print("\n")
        print(" REMOVE SONG FROM QUEUE")
        print(f"Current queue size: {len(self.queue)}\n")
        
        for i, song in enumerate(self.queue, 1):
            print(f"{i:2d}. {song}")
        
        try:
            choice = input("\nEnter song number to remove (or 0 to cancel): ").strip()
            if choice == '0':
                return
            
            idx = int(choice) - 1
            if 0 <= idx < len(self.queue):
                removed = self.queue.pop(idx)
                print(f"\n Removed: {removed}")
                print(f" Queue size: {len(self.queue)}")
            else:
                print("\n Invalid song number!")
        except ValueError:
            print("\n Please enter a valid number!")
    
    def view_queue_and_recommendations(self):
        
        print("\n")
        print(f" CURRENT QUEUE ({len(self.queue)} songs)")
        
        if not self.queue:
            print("Queue is empty! Add songs to get started.")
            return
        
        for i, song in enumerate(self.queue, 1):
            print(f"{i:2d}. {song}")
        
        # Get recommendations
        print("\n")
        print(" RECOMMENDED SONGS (Top 5)")
        
        recommendations = recommend_songs(self.queue, self.song_names, self.feature_matrix, k=5)
        
        if not recommendations:
            print("No recommendations available.")
            return
        
        for i, (song, score) in enumerate(recommendations, 1):
            print(f"{i}. {song}")
            print(f"    Similarity Score: {score:.4f}")
    
    def clear_queue(self):
        
        if len(self.queue) == 0:
            print("\n  Queue is already empty!")
            return
        
        confirm = input(f"\n  Are you sure you want to clear all {len(self.queue)} songs? (y/n): ").strip().lower()
        if confirm == 'y':
            self.queue = []
            print("\n Queue cleared successfully!")
        else:
            print("\n Clear operation cancelled.")
    
    def view_all_songs(self):
       
        print("\n")
        print(f" ALL AVAILABLE SONGS ({len(self.all_songs_list)} total)")
        
        print("\nOptions:")
        print("  1. View all songs")
        print("  2. View by page (20 songs per page)")
        print("  0. Go back")
        
        choice = input("\nYour choice: ").strip()
        
        if choice == '1':
            for i, song in enumerate(self.all_songs_list, 1):
                status = "[IN QUEUE]" if song in self.queue else ""
                print(f"{i:3d}. {song} {status}")
        
        elif choice == '2':
            self.view_songs_paginated()
    
    def view_songs_paginated(self):
        
        page_size = 20
        total_pages = (len(self.all_songs_list) + page_size - 1) // page_size
        current_page = 1
        
        while True:
            start_idx = (current_page - 1) * page_size
            end_idx = min(start_idx + page_size, len(self.all_songs_list))
            
            print(f"\n--- Page {current_page}/{total_pages} ---")
            for i in range(start_idx, end_idx):
                status = "[IN QUEUE]" if self.all_songs_list[i] in self.queue else ""
                print(f"{i+1:3d}. {self.all_songs_list[i]} {status}")
            
            print(f"\nOptions: [N]ext, [P]revious, [Q]uit")
            choice = input("Your choice: ").strip().lower()
            
            if choice == 'n' and current_page < total_pages:
                current_page += 1
            elif choice == 'p' and current_page > 1:
                current_page -= 1
            elif choice == 'q':
                break
    
    def search_song(self):
        
        print("\n" )
        print(" SEARCH FOR SONGS")
        
        keyword = input("Enter search keyword (song name or artist): ").strip().lower()
        
        if not keyword:
            print(" Please enter a search term!")
            return
        
        results = [song for song in self.all_songs_list if keyword in song.lower()]
        
        if not results:
            print(f"\n No songs found matching '{keyword}'")
            return
        
        print(f"\n Found {len(results)} song(s):\n")
        for i, song in enumerate(results, 1):
            status = "[IN QUEUE]" if song in self.queue else ""
            print(f"{i:2d}. {song} {status}")
        
        # Option to add from search results
        add_choice = input("\nAdd a song from results? (enter number or 0 to skip): ").strip()
        try:
            if add_choice != '0':
                idx = int(add_choice) - 1
                if 0 <= idx < len(results):
                    self.add_song(results[idx])
        except ValueError:
            pass
    
    def view_detailed_analysis(self):
        
        if len(self.queue) == 0:
            print("\n Queue is empty! Add songs first to see analysis.")
            return
        
        print("\n")
        print(" DETAILED MATHEMATICAL ANALYSIS")
        
        centroid = compute_centroid(self.queue, self.song_names, self.feature_matrix)
        
        print(f"\n Queue Statistics:")
        print(f"  • Number of songs: {len(self.queue)}")
        print(f"  • Feature dimensions: {len(centroid)}")
        print(f"  • Centroid computed: ")
        
        print(f"\n Dominant Features (>0.2 threshold):")
        prominent_features = [(feature_columns[i], val) for i, val in enumerate(centroid) if val > 0.2]
        
        if prominent_features:
            for feature, value in sorted(prominent_features, key=lambda x: x[1], reverse=True):
                print(f"  • {feature}: {value:.3f}")
        else:
            print("  No dominant features found.")
        
        # Show top 10 recommendations with scores
        print(f"\n Top 10 Recommendations with Similarity Scores:")
        recommendations = recommend_songs(self.queue, self.song_names, self.feature_matrix, k=10)
        
        for i, (song, score) in enumerate(recommendations, 1):
            print(f"{i:2d}. {song[:45]:45s} → {score:.4f}")
    
    def create_random_queue(self):
       
        print("\n")
        print(" CREATE RANDOM QUEUE")
        
        try:
            num_songs = input("How many random songs? (1-20): ").strip()
            num_songs = int(num_songs)
            
            if num_songs < 1 or num_songs > 20:
                print(" Please enter a number between 1 and 20!")
                return
            
            if num_songs > len(self.all_songs_list):
                num_songs = len(self.all_songs_list)
            
            self.queue = list(np.random.choice(self.all_songs_list, size=num_songs, replace=False))
            print(f"\n Created random queue with {num_songs} songs!")
            self.view_queue_and_recommendations()
            
        except ValueError:
            print(" Please enter a valid number!")
    
    def load_preset_queue(self):
       
        print("\n")
        print(" LOAD PRESET QUEUE")
        
        print("\nAvailable presets:")
        print("  1. Assignment Seed Queue (10 songs)")
        print("  2. Pop Favorites (5 songs)")
        print("  3. Rock Classics (5 songs)")
        print("  0. Cancel")
        
        choice = input("\nYour choice: ").strip()
        
        presets = {
            '1': [
                "Call Out My Name - The Weeknd",
                "Hotel California - Eagles",
                "Bohemian Rhapsody - Queen",
                "Toxic - Britney Spears",
                "Smells Like Teen Spirit - Nirvana",
                "Halo - Beyoncé",
                "One More Time - Daft Punk",
                "Stairway to Heaven - Led Zeppelin",
                "Dancing Queen - ABBA",
                "Enter Sandman - Metallica"
            ],
            '2': [
                "Shake It Off - Taylor Swift",
                "Shape of You - Ed Sheeran",
                "Hello - Adele",
                "Thank U Next - Ariana Grande",
                "Bad Guy - Billie Eilish"
            ],
            '3': [
                "Stairway to Heaven - Led Zeppelin",
                "Bohemian Rhapsody - Queen",
                "Hotel California - Eagles",
                "Smells Like Teen Spirit - Nirvana",
                "Enter Sandman - Metallica"
            ]
        }
        
        if choice in presets:
            # Verify all songs exist
            valid_songs = [s for s in presets[choice] if s in self.song_names]
            if len(valid_songs) != len(presets[choice]):
                print(f"\n  Warning: Some preset songs not found in dataset. Loading {len(valid_songs)} songs.")
            
            self.queue = valid_songs
            print(f"\n Loaded preset queue with {len(self.queue)} songs!")
            self.view_queue_and_recommendations()
        elif choice == '0':
            return
        else:
            print(" Invalid choice!")
    
    def run(self):
        
        print("\n")
        print(" Welcome to the Interactive Music Recommendation System!")
        
        while True:
            self.display_menu()
            choice = input("\n Enter your choice (0-9): ").strip()
            
            if choice == '1':
                self.add_song_interactive()
            
            elif choice == '2':
                self.remove_song_interactive()
            
            elif choice == '3':
                self.view_queue_and_recommendations()
            
            elif choice == '4':
                self.clear_queue()
            
            elif choice == '5':
                self.view_all_songs()
            
            elif choice == '6':
                self.search_song()
            
            elif choice == '7':
                self.view_detailed_analysis()
            
            elif choice == '8':
                self.create_random_queue()
            
            elif choice == '9':
                self.load_preset_queue()
            
            elif choice == '0':
                print("\n")
                print(" Thank you for using the Music Recommendation System!")
                break
            
            else:
                print("\n Invalid choice! Please enter a number between 0 and 9.")
            
            # Pause before showing menu again
            input("\nPress Enter to continue...")

print(" InteractiveMusicQueue class defined!")

# LAUNCH THE INTERACTIVE SYSTEM


In [ ]:
# Create and run the interactive system
music_system = InteractiveMusicQueue(song_names, feature_matrix)
music_system.run()